In [1]:
!pip install pyarrow


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
import sklearn
print(sklearn.__version__)

1.8.0


In [6]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [7]:
mean_squared_error

<function sklearn.metrics._regression.mean_squared_error(y_true, y_pred, *, sample_weight=None, multioutput='uniform_average')>

In [39]:
import os, mlflow, pathlib

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_registry_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment2")



<Experiment: artifact_location='file:C:/Users/Eriton/source/repos/mlops-zoomcamp/02-model-registry/experiment_tracking/mlruns/3', creation_time=1776814250416, experiment_id='3', last_update_time=1776814250416, lifecycle_stage='active', name='nyc-taxi-experiment2', tags={}, trace_location=None, workspace='default'>

In [9]:
pd.__version__

'2.3.3'

In [10]:
def read_dataframe(filename):

    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [11]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')


In [12]:
len(df_train), len(df_val)

(73908, 61921)

In [13]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [14]:
df_val

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration,PU_DO
0,2,2021-02-01 00:34:03,2021-02-01 00:51:58,N,1.0,130,205,5.0,3.66,14.00,...,10.00,0.0,None,0.3,25.30,1.0,1.0,0.00,17.916667,130_205
1,2,2021-02-01 00:04:00,2021-02-01 00:10:30,N,1.0,152,244,1.0,1.10,6.50,...,0.00,0.0,None,0.3,7.80,2.0,1.0,0.00,6.500000,152_244
2,2,2021-02-01 00:18:51,2021-02-01 00:34:06,N,1.0,152,48,1.0,4.93,16.50,...,0.00,0.0,None,0.3,20.55,2.0,1.0,2.75,15.250000,152_48
3,2,2021-02-01 00:53:27,2021-02-01 01:11:41,N,1.0,152,241,1.0,6.70,21.00,...,0.00,0.0,None,0.3,22.30,2.0,1.0,0.00,18.233333,152_241
4,2,2021-02-01 00:57:46,2021-02-01 01:06:44,N,1.0,75,42,1.0,1.89,8.50,...,2.45,0.0,None,0.3,12.25,1.0,1.0,0.00,8.966667,75_42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64567,2,2021-02-28 22:19:00,2021-02-28 22:29:00,None,NaN,129,7,NaN,2.63,10.04,...,0.00,0.0,None,0.3,10.34,NaN,NaN,NaN,10.000000,129_7
64568,2,2021-02-28 23:18:00,2021-02-28 23:27:00,None,NaN,116,166,NaN,1.87,8.33,...,1.89,0.0,None,0.3,10.52,NaN,NaN,NaN,9.000000,116_166
64569,2,2021-02-28 23:44:00,2021-02-28 23:58:00,None,NaN,74,151,NaN,2.40,12.61,...,0.00,0.0,None,0.3,12.91,NaN,NaN,NaN,14.000000,74_151
64570,2,2021-02-28 23:07:00,2021-02-28 23:14:00,None,NaN,42,42,NaN,1.11,11.95,...,0.00,0.0,None,0.3,15.00,NaN,NaN,NaN,7.000000,42_42


In [15]:
categorical = ['PU_DO']   #  ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)


In [16]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [17]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

#mean_squared_error(y_val, y_pred, squared=False)
root_mean_squared_error(y_val, y_pred)

7.758715202840848

In [18]:
mlflow.sklearn.autolog(log_datasets=False)

In [19]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [40]:
with mlflow.start_run():
    
    mlflow.set_tag("developer","eriton")
    
    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    # log the preprocessor DictVectorizer
    with open("models/preprocessor.bin", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.bin", artifact_path="preprocessor")

    rmse = root_mean_squared_error(y_val, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    
    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

2026/04/21 21:53:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [21]:
import xgboost as xgb

In [22]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [23]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [24]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [25]:
# reg:linear is deprecated use reg:squarederror

search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:11.58411                                                                   
[1]	validation-rmse:11.02111                                                                   
  0%|                                                   | 0/50 [00:00<?, ?trial/s, best loss=?]

C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [20:40:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[2]	validation-rmse:10.51675                                                                   
[3]	validation-rmse:10.06594                                                                   
[4]	validation-rmse:9.66505                                                                    
[5]	validation-rmse:9.30987                                                                    
[6]	validation-rmse:8.99461                                                                    
[7]	validation-rmse:8.71637                                                                    
[8]	validation-rmse:8.47058                                                                    
[9]	validation-rmse:8.25451                                                                    
[10]	validation-rmse:8.06374                                                                   
[11]	validation-rmse:7.89738                                                                   
[12]	validation-rmse:7.75011            

C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [20:40:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:11.69031                                                                   
[1]	validation-rmse:11.21305                                                                   
[2]	validation-rmse:10.77679                                                                   
[3]	validation-rmse:10.37860                                                                   
[4]	validation-rmse:10.01585                                                                   
[5]	validation-rmse:9.68616                                                                    
[6]	validation-rmse:9.38738                                                                    
[7]	validation-rmse:9.11562                                                                    
[8]	validation-rmse:8.87144                                                                    
[9]	validation-rmse:8.64954                                                                    
[10]	validation-rmse:8.44957            

...

[0]	validation-rmse:11.76937                                                                   
[1]	validation-rmse:11.35678                                                                   
[2]	validation-rmse:10.97406                                                                   
[3]	validation-rmse:10.61910                                                                   
[4]	validation-rmse:10.29027                                                                   
[5]	validation-rmse:9.98632                                                                    
[6]	validation-rmse:9.70536                                                                    
[7]	validation-rmse:9.44628                                                                    
[8]	validation-rmse:9.20744                                                                    
[9]	validation-rmse:8.98747                                                                    
[10]	validation-rmse:8.78496            

C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [21:29:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:11.55324                                                                   
[1]	validation-rmse:10.96434                                                                   
[2]	validation-rmse:10.43958                                                                   
[3]	validation-rmse:9.97174                                                                    
[4]	validation-rmse:9.55969                                                                    
[5]	validation-rmse:9.19619                                                                    
[6]	validation-rmse:8.87247                                                                    
[7]	validation-rmse:8.58741                                                                    
[8]	validation-rmse:8.33830                                                                    
[9]	validation-rmse:8.12291                                                                    
[10]	validation-rmse:7.92795            

C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [21:30:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:11.25770                                                                   
[1]	validation-rmse:10.45338                                                                   
[2]	validation-rmse:9.77979                                                                    
[3]	validation-rmse:9.22008                                                                    
[4]	validation-rmse:8.75748                                                                    
[5]	validation-rmse:8.37794                                                                    
[6]	validation-rmse:8.06692                                                                    
[7]	validation-rmse:7.81365                                                                    
[8]	validation-rmse:7.60679                                                                    
[9]	validation-rmse:7.43914                                                                    
[10]	validation-rmse:7.30359            

In [26]:
params = { 
    'max_depth': 41,
    'learning_rate': 0.08048724945407973,
    'reg_alpha': 0.011688402031886633,
    'reg_lambda': 0.004836768546153373,
    'min_child_weight': 1.2306633622264325,
    'objective': 'reg:linear',
    'seed': 42
}

In [27]:
mlflow.xgboost.autolog()

booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )

2026/04/21 21:33:33 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '45d8a0d3a2fb47a38111899b7a5d9cc9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow
C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [21:33:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.56274
[1]	validation-rmse:10.97980
[2]	validation-rmse:10.45918
[3]	validation-rmse:9.99309
[4]	validation-rmse:9.58068
[5]	validation-rmse:9.21196
[6]	validation-rmse:8.88714
[7]	validation-rmse:8.60014
[8]	validation-rmse:8.34798
[9]	validation-rmse:8.12540
[10]	validation-rmse:7.92995
[11]	validation-rmse:7.75993
[12]	validation-rmse:7.60817
[13]	validation-rmse:7.47737
[14]	validation-rmse:7.36125
[15]	validation-rmse:7.26119
[16]	validation-rmse:7.17201
[17]	validation-rmse:7.09353
[18]	validation-rmse:7.02673
[19]	validation-rmse:6.96606
[20]	validation-rmse:6.91461
[21]	validation-rmse:6.86840
[22]	validation-rmse:6.82805
[23]	validation-rmse:6.79238
[24]	validation-rmse:6.76135
[25]	validation-rmse:6.73299
[26]	validation-rmse:6.70765
[27]	validation-rmse:6.68506
[28]	validation-rmse:6.66566
[29]	validation-rmse:6.64737
[30]	validation-rmse:6.63176
[31]	validation-rmse:6.61735
[32]	validation-rmse:6.60405
[33]	validation-rmse:6.59233
[34]	validation-rmse:

2026/04/21 21:34:49 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/04/21 21:34:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [30]:
logged_model = 'runs:/45d8a0d3a2fb47a38111899b7a5d9cc9/model'

In [31]:
# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)


   

In [32]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: file:C:/Users/Eriton/source/repos/mlops-zoomcamp/02-model-registry/experiment_tracking/mlruns/3/models/m-396beabe86b64015b7b0d6d77cad52a6/artifacts
  flavor: mlflow.xgboost
  run_id: 45d8a0d3a2fb47a38111899b7a5d9cc9

In [33]:
valid = xgb.DMatrix(X_val, label=y_val)

In [34]:
loaded_model.predict(X_val)

array([14.527894 ,  7.1385155, 15.74355  , ..., 13.456723 ,  6.3960595,
        8.215588 ], dtype=float32)

In [35]:
xgb_model = mlflow.xgboost.load_model(logged_model)
xgb_model

In [36]:
xgb_model.predict(valid)

array([14.527894 ,  7.1385155, 15.74355  , ..., 13.456723 ,  6.3960595,
        8.215588 ], dtype=float32)

In [37]:
mlflow.xgboost.autolog(disable=True)
mlflow.set_experiment("my-cool-experiment")
with mlflow.start_run():
# Hyperparameter for run 09923bbad64045ca837a1656254ce756
    params = { 
        'max_depth': 41,
        'learning_rate': 0.08048724945407973,
        'reg_alpha': 0.011688402031886633,
        'reg_lambda': 0.004836768546153373,
        'min_child_weight': 1.2306633622264325,
        'objective': 'reg:linear',
        'seed': 42
    }
    mlflow.log_params(params)
    
    booster = xgb.train(
                params=params,
                dtrain=train,
                num_boost_round=1000,
                evals=[(valid, 'validation')],
                early_stopping_rounds=50
            )
    # get the y_pred from X_train
    y_pred = booster.predict(valid)
    # get RMSE and record on mlflow
    rmse = round(root_mean_squared_error(y_val, y_pred),2)
    print("RMSE for training data:", rmse)
    mlflow.log_metric("rmse", rmse)
    # log xgboost model to mlflow
    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")
    # log the preprocessor DictVectorizer
    with open("models/preprocessor.bin", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.bin", artifact_path="preprocessor")

C:\Users\Eriton\source\repos\mlops-zoomcamp\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [21:38:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.56274
[1]	validation-rmse:10.97980
[2]	validation-rmse:10.45918
[3]	validation-rmse:9.99309
[4]	validation-rmse:9.58068
[5]	validation-rmse:9.21196
[6]	validation-rmse:8.88714
[7]	validation-rmse:8.60014
[8]	validation-rmse:8.34798
[9]	validation-rmse:8.12540
[10]	validation-rmse:7.92995
[11]	validation-rmse:7.75993
[12]	validation-rmse:7.60817
[13]	validation-rmse:7.47737
[14]	validation-rmse:7.36125
[15]	validation-rmse:7.26119
[16]	validation-rmse:7.17201
[17]	validation-rmse:7.09353
[18]	validation-rmse:7.02673
[19]	validation-rmse:6.96606
[20]	validation-rmse:6.91461
[21]	validation-rmse:6.86840
[22]	validation-rmse:6.82805
[23]	validation-rmse:6.79238
[24]	validation-rmse:6.76135
[25]	validation-rmse:6.73299
[26]	validation-rmse:6.70765
[27]	validation-rmse:6.68506
[28]	validation-rmse:6.66566
[29]	validation-rmse:6.64737
[30]	validation-rmse:6.63176
[31]	validation-rmse:6.61735
[32]	validation-rmse:6.60405
[33]	validation-rmse:6.59233
[34]	validation-rmse:

2026/04/21 21:39:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE for training data: 6.31
